In [ ]:
import requests
import json
import time
from datetime import datetime, timedelta

auth = json.loads(mssparkutils.notebook.run("procore_auth"))
token = auth["token"]
COMPANY_ID = auth["company_id"]
headers = {
    "Authorization": f"Bearer {token}",
    "Procore-Company-Id": str(COMPANY_ID)
}

print("Auth successful" if token else "Auth failed")

StatementMeta(, f11e395b-aa54-4c1d-807a-b2bec79d45b0, 3, Finished, Available, Finished, False)

Auth successful


In [2]:
projects_response = requests.get(
    "https://api.procore.com/rest/v1.0/projects",
    headers=headers,
    params={"company_id": COMPANY_ID}
)
projects = projects_response.json()
print(f"{len(projects)} projects found" if isinstance(projects, list) else "Failed to fetch projects")

StatementMeta(, f11e395b-aa54-4c1d-807a-b2bec79d45b0, 4, Finished, Available, Finished, False)

18 projects found


In [3]:
all_payment_apps = []

for project in projects:
    project_id = project["id"]
    project_name = project["name"]

    page = 1
    while True:
        response = requests.get(
            "https://api.procore.com/rest/v1.0/payment_applications",
            headers=headers,
            params={
                "project_id": project_id,
                "page": page,
                "per_page": 100
            }
        )

        if response.status_code != 200:
            break

        rows = response.json()

        if not rows or isinstance(rows, dict):
            break

        for row in rows:
            row["project_id"] = project_id
            row["project_name"] = project_name

        all_payment_apps.extend(rows)

        if len(rows) < 100:
            break

        page += 1
        time.sleep(0.3)

print(f"Total payment applications: {len(all_payment_apps)}")

StatementMeta(, f11e395b-aa54-4c1d-807a-b2bec79d45b0, 5, Finished, Available, Finished, False)

Total payment applications: 124


In [4]:
all_line_items = []

print(f"Pulling g703 line items for {len(all_payment_apps)} payment applications...")

for app in all_payment_apps:
    app_id = app["id"]
    project_id = app["project_id"]
    project_name = app["project_name"]
    billing_date = app.get("billing_date", "")
    invoice_number = app.get("invoice_number", "")
    app_number = app.get("number", "")
    app_status = app.get("status", "")
    period_start = app.get("period_start", "")
    period_end = app.get("period_end", "")
    total_amount_accrued = app.get("total_amount_accrued_this_period", "")
    contract = app.get("contract", {})
    prime_contract_id = contract.get("id", "") if isinstance(contract, dict) else ""

    max_retries = 3
    retry_count = 0

    while retry_count < max_retries:
        response = requests.get(
            f"https://api.procore.com/rest/v1.0/payment_applications/{app_id}",
            headers=headers,
            params={"project_id": project_id}
        )

        if response.status_code == 429:
            print(f"  Rate limited — waiting 60 seconds...")
            time.sleep(60)
            retry_count += 1
            continue

        if response.status_code != 200:
            print(f"  Error {response.status_code} for payment app {app_id}, skipping")
            break

        data = response.json()
        g703 = data.get("g703", [])

        if not g703:
            break

        for item in g703:
            cost_code = item.get("cost_code", {})
            wbs_code = item.get("wbs_code", {})

            all_line_items.append({
                "payment_application_id": app_id,
                "payment_application_number": app_number,
                "payment_application_status": app_status,
                "invoice_number": invoice_number,
                "billing_date": billing_date,
                "period_start": period_start,
                "period_end": period_end,
                "total_amount_accrued_this_period": total_amount_accrued,
                "prime_contract_id": prime_contract_id,
                "project_id": project_id,
                "project_name": project_name,
                "line_item_id": item.get("id"),
                "item_number": item.get("item_number"),
                "cost_code_id": cost_code.get("id") if isinstance(cost_code, dict) else None,
                "cost_code_full_code": cost_code.get("full_code") if isinstance(cost_code, dict) else None,
                "cost_code_name": cost_code.get("name") if isinstance(cost_code, dict) else None,
                "wbs_flat_code": wbs_code.get("flat_code") if isinstance(wbs_code, dict) else None,
                "wbs_description": wbs_code.get("description") if isinstance(wbs_code, dict) else None,
                "description_of_work": item.get("description_of_work"),
                "scheduled_value": item.get("scheduled_value"),
                "work_completed_from_previous_application": item.get("work_completed_from_previous_application"),
                "work_completed_this_period": item.get("work_completed_this_period"),
                "total_completed_and_stored_to_date": item.get("total_completed_and_stored_to_date"),
                "total_completed_and_stored_to_date_percent": item.get("total_completed_and_stored_to_date_percent"),
                "materials_presently_stored": item.get("materials_presently_stored"),
                "balance_to_finish": item.get("balance_to_finish"),
                "total_retainage_currently_retained": item.get("total_retainage_currently_retained"),
                "work_completed_retainage_retained_this_period": item.get("work_completed_retainage_retained_this_period"),
                "work_completed_retainage_released_this_period": item.get("work_completed_retainage_released_this_period"),
            })
        break

    time.sleep(2)

print(f"Done! Total line items: {len(all_line_items)}")

StatementMeta(, f11e395b-aa54-4c1d-807a-b2bec79d45b0, 6, Finished, Available, Finished, False)

Pulling g703 line items for 124 payment applications...
Done! Total line items: 7699


In [5]:
import pandas as pd
import re

def clean_column_name(col):
    col = col.strip()
    col = re.sub(r'[ ,;{}()\n\t=]', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    return col

pdf = pd.DataFrame(all_line_items)
pdf.columns = [clean_column_name(c) for c in pdf.columns]

for col in pdf.columns:
    if pdf[col].dtype == object:
        pdf[col] = pdf[col].astype(str).replace('None', None)

spark.sql("DROP TABLE IF EXISTS procore_payment_application_line_items_raw")

df = spark.createDataFrame(pdf)
df.write.format("delta").mode("append").saveAsTable("procore_payment_application_line_items_raw")

print("Saved to Bronze_Lakehouse successfully")

StatementMeta(, f11e395b-aa54-4c1d-807a-b2bec79d45b0, 7, Finished, Available, Finished, False)

Saved to Bronze_Lakehouse successfully
